# Camera stream — Tang Nano 4K

Receives frames from the Tang Nano 4K's OV2640 one after another, through the 8 MB PSRAM (HyperRAM) inside
the FPGA package. Everything goes over the one USB cable (JTAG through the onboard debugger).

The overlay has three general-purpose IPs next to the base design (LED, buttons):
- `cam_i2c`: an I2C master (the camera's SCCB configuration bus)
- `cam_dvp`: the camera clock (XCLK) and the pixel front end
- `cam_ring`: a frame ring in the PSRAM: 256 slots of 32 KB, one frame each, read through a 1 KB window

The ring takes every frame at the sensor's rate (about 8.7 frames/s at 160x120 grayscale). The host reads
them over JTAG at about 48 kB/s, so about 2 frames/s; when the ring is full, new frames are dropped and
counted. `stream()` reads frames as fast as the host can; `record(n)` lets n frames pile up at the sensor's
rate, then reads them all.

Before you start, see `02_led_tangnano4k.ipynb` (Zadig,
`python -m nanopynq doctor --board tangnano4k`).

## 1. Load the camera stream overlay

In [ ]:
from nanopynq import Overlay
from nanopynq.camera import Camera

ol = Overlay("camstream", board="tangnano4k")
ol

## 2. Put the camera together and start it

There is no capture buffer in this overlay (its block RAM holds the ring's FIFO and window), so the second
argument is `None` and the ring goes in `ring=`. `start()` is the same as for snapshots (about 1.5 s).

In [ ]:
cam = Camera(ol.cam_i2c, None, ol.cam_dvp, ring=ol.cam_ring)
cam.start()

## 3. Stream

Each frame replaces the previous one in the output below. Frames the ring marked bad (its write FIFO
overflowed, or the frame did not fit in a slot) are skipped and counted in `cam.bad`.

In [ ]:
import itertools
import time

from IPython.display import display

view = display(None, display_id=True)
t0 = time.perf_counter()
for f in cam.stream(20):
    view.update(f)
dt = time.perf_counter() - t0
print(
    f"20 frames in {dt:.1f} s ({20 / dt:.2f} frames/s), last seq {f.seq}, dropped {ol.cam_ring.dropped}, bad {cam.bad}"
)

## 4. Record at the sensor's rate

`record(50)` turns the ring on, waits until 50 frames are in the PSRAM (about 6 s at 8.7 frames/s), turns
it off and reads them (about 0.45 s each). Consecutive sequence numbers mean no frame was lost.

In [ ]:
t0 = time.perf_counter()
frames = cam.record(50)
print(
    f"{len(frames)} frames in {time.perf_counter() - t0:.1f} s, seq {frames[0].seq}..{frames[-1].seq}, "
    f"gaps {[b.seq - a.seq for a, b in itertools.pairwise(frames) if b.seq != a.seq + 1]}, "
    f"dropped {ol.cam_ring.dropped}, bad {cam.bad}"
)
for f in frames[::16]:
    display(f)

## 5. The registers underneath

`WSEQ` counts the frames written, `RSEQ` the frames read (the host advances it), `DROPPED` the frames
thrown away while the ring was full; `SLOT_LEN` is the bytes of the last frame. `FRAMES` (DVP) counts the
sensor's frames, `PCLKS` and `LINES` describe the last one.

In [ ]:
ol.cam_ring.register_map, {"frames": ol.cam_dvp.frames, "pclks": ol.cam_dvp.pclks, "lines": ol.cam_dvp.lines}

## 6. Close

In [ ]:
ol.close()